# Reading the Raw Manifestos

The goal of this notebook is simple: read the manifesto PDFs stored in GitHub and organize their extracted text as a corpus.

We will not clean or transform the text yet.

> **Final structure:** one row = one party manifesto.


## 1. Find the source files

The manifesto PDFs are stored in the `RAW` folder of a GitHub repository. Instead of typing every filename manually, we ask GitHub which files are available in that folder.

### AI prompt

> I have several PDF files in the `RAW` folder of this GitHub repository: `https://github.com/eScience-SummerSchool/manifestos`. I do not want to clone the repository. Write simple Python code to use the GitHub API to identify the files available in that folder.


In [ ]:
import requests

api_url = "https://api.github.com/repos/eScience-SummerSchool/manifestos/contents/RAW"

files = requests.get(api_url).json()

files


## 2. Identify the PDF files

The folder may contain files other than manifestos. We keep only filenames ending in `.pdf` and inspect them before continuing.

### AI prompt

> The GitHub API returned a list called `files`, where each item contains metadata including the filename in `file["name"]`. Write simple Python code to keep only PDF files and show their names.


In [ ]:
pdf_files = [
    file for file in files
    if file["name"].lower().endswith(".pdf")
]

[file["name"] for file in pdf_files]


## 3. Get the direct location of each PDF

GitHub provides a `download_url` for every file. We collect those URLs so the same procedure can be applied to every manifesto.

### AI prompt

> I have a list called `pdf_files` containing GitHub metadata for PDF files. Each item has a `download_url`. Create a list called `pdf_urls` containing only those URLs.


In [ ]:
pdf_urls = [file["download_url"] for file in pdf_files]

pdf_urls


## 4. Choose a tool to read PDF text

Python needs a library that can open PDFs and extract their machine-readable text. Here we use **PyMuPDF**.

### AI prompt

> I need to extract machine-readable text from several PDF files in Python or Google Colab. I do not need OCR. Suggest a simple and reliable library and show me how to install and import it.


In [ ]:
# Run this only if PyMuPDF is not already installed
# !pip install pymupdf

import pymupdf as fitz


## 5. Extract the manifesto texts

Each PDF contains several pages. We read each PDF from its URL, extract the text page by page, combine those pages into one complete document, and use the PDF filename as the party name.

The result will be a dictionary:

**party → complete manifesto text**

### AI prompt

> I have a list called `pdf_urls` containing direct URLs to PDF files. Using `requests` and PyMuPDF, read each PDF into memory, extract its text page by page, derive the party name from the PDF filename, and store the results in a dictionary called `documents`, where the party is the key and the complete text is the value. Keep the code explicit and easy to read.


In [ ]:
documents = {}

for pdf_url in pdf_urls:

    # Read the PDF from its GitHub URL into memory
    pdf_data = requests.get(pdf_url).content

    # Open the PDF so PyMuPDF can read its pages
    doc = fitz.open(stream=pdf_data, filetype="pdf")

    # Start an empty text for this manifesto
    text = ""

    # Visit every page and add its text to the manifesto
    for page in doc:
        text += page.get_text()

    # Use the PDF filename as the party name
    # Example: .../BuenGobierno.pdf -> BuenGobierno
    party = pdf_url.split("/")[-1].replace(".pdf", "")

    # Store the complete text under the party name
    documents[party] = text


## 6. Verify the extracted documents

Before creating the corpus, we check which party names were produced.

### AI prompt

> I have a Python dictionary called `documents`, where the keys should be party names and the values are manifesto texts. What is the simplest way to check which parties were extracted?


In [ ]:
documents.keys()


## 7. Create the corpus

A dictionary is convenient for extraction, but a DataFrame is easier to analyze. We convert `documents` into a corpus where each row represents one manifesto.

The first variables are:

- `party`
- `text_raw`

### AI prompt

> I have a dictionary called `documents` with the structure `{party: complete_text}`. Convert it into a Pandas DataFrame called `corpus` with one manifesto per row and columns called `party` and `text_raw`.


In [ ]:
import pandas as pd

corpus = (
    pd.DataFrame.from_dict(
        documents,
        orient="index",
        columns=["text_raw"]
    )
    .reset_index(names="party")
)

corpus


## 8. Describe the size of the documents

Before cleaning the texts, we calculate two simple characteristics: the number of characters and an approximate number of words.

The word count here is based only on whitespace. It is **not yet linguistic tokenization**.

### AI prompt

> My Pandas DataFrame `corpus` contains one manifesto per row and the raw text is stored in `text_raw`. Using Pandas string methods, add `n_characters` and a simple whitespace-based `n_words` count.


In [ ]:
corpus["n_characters"] = corpus["text_raw"].str.len()
corpus["n_words"] = corpus["text_raw"].str.split().str.len()

corpus[["party", "n_characters", "n_words"]]


## 9. Save the raw corpus

The extraction stage is complete. We save the corpus so later notebooks can start from the structured text rather than reading the PDFs again.

The word **raw** means that the text has been extracted and organized, but not cleaned.

### AI prompt

> Save my Pandas DataFrame `corpus` as `corpus_raw.csv` without writing the DataFrame index to the file.


In [ ]:
corpus.to_csv("corpus_raw.csv", index=False)


## Result

The workflow is now:

**GitHub RAW folder → PDF URLs → extracted text → documents dictionary → corpus DataFrame → `corpus_raw.csv`**

No text cleaning has been performed yet.
